- Here we have learned application of `output_parsers` with both `HuggingFace` (these can't work with with_structured_output) and `OpenAI` models (these can work with with_structured_output).
- Output parsers can work with both types.
- Basically output parser gives a structure according to type of output parser to the output coming from llm.
- Here we will see difference in output when we will use output parser vs not use it.
- `TypedDict`, `Pydantic` or `Json schema` enforcement will be not here, only we will use output parser to give output different forms.
- as huggingface models can work with none of `TypedDict`, `Pydantic` or `Json schema`, leading to mismatched for `with_structured_output` function, so We are learning output parser to get generilized idea of structuring and validating output from LLM.


## StrOutputParser
- This will just extract content from output of LLM which contains result and metadata both.
- There is no further change or structuring in it.

### HuggingFace Model

In [3]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate


load_dotenv()


# can try with "TinyLlama/TinyLlama-1.1B-chat-v1.0" as well
llm = HuggingFaceEndpoint(
    repo_id = "google/gemma-2-2b-it",
    task = "text-generation"
)

model = ChatHuggingFace(llm=llm)

# 1st prompt -> detailed report
template1 = PromptTemplate(
    template = "Write a detailed report on {topic}",
    input_variables = ['topic']
)

# 2nd prompt -> summary
template2 = PromptTemplate(
    template = "Write a five line summary on the following text. \n {text}",
    input_variables = ['text']
)


prompt1 = template1.invoke({'topic': 'black hole'}) # generation of first prompt
result = model.invoke(prompt1) # lllm's reply on first prompt
prompt2 = template2.invoke({
    'text': result.content
})

result1 = model.invoke(prompt2)
print(result1.content)

BadRequestError: (Request ID: Root=1-6a4a44bf-683141871e6e2c5727abd08e;ca804f5a-4966-4ada-b28d-9cb8a88ecd22)

Bad request:
{'message': "The requested model 'google/gemma-2-2b-it' is not supported by any provider you have enabled.", 'type': 'invalid_request_error', 'param': 'model', 'code': 'model_not_supported'}

- Unable to check the HuggingFace model as facing issue in loading it.

### OpenAI Model

In [ ]:

from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate

load_dotenv()

model = ChatOpenAI()

# 1st prompt -> detailed report
template1 = PromptTemplate(
    template = "Write a detailed report on {topic}",
    input_variables = ['topic']
)

# 2nd prompt -> summary
template2 = PromptTemplate(
    template = "Write a five line summary on the following text. \n {text}",
    input_variables = ['text']
)

# generation of first prompt
prompt1 = template1.invoke({'topic': 'black hole'}) 

# lllm's reply on first prompt
result = model.invoke(prompt1) 

# generation of second prompt
prompt2 = template2.invoke({
    'text': result.content
})

# lllm's reply on second prompt
result1 = model.invoke(prompt2)

# print only content part (excluding metadata)
print(result1.content)

Black holes are regions in space with intense gravitational pull that not even light can escape from, formed when massive stars collapse into a singularity. There are three types of black holes based on their mass: stellar, supermassive, and intermediate. Black holes distort space-time, emit Hawking radiation, and were first theorized by John Michell with their existence confirmed in the 20th century. Ongoing research aims to understand the physics of black holes and their role in the universe, with innovations like the Event Horizon Telescope providing new insights into their structure and properties.


In [ ]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser


load_dotenv()


model = ChatOpenAI()

# 1st prompt -> detailed report
template1 = PromptTemplate(
    template = "Write a detailed report on {topic}",
    input_variables = ['topic']
)

# 2nd prompt -> summary
template2 = PromptTemplate(
    template = "Write a five line summary on the following text. \n {text}",
    input_variables = ['text']
)


parser = StrOutputParser()

# do full process via this chain, it is possible because of this parser here
chain = template1 | model | parser | template2 | model | parser

result = chain.invoke({'topic': 'black hole'})

# as we are using parser, so `result` will contain no metadata 
print(result)

Black holes are regions in space with strong gravitational pull, first proposed by John Michell and elaborated on by Albert Einstein. They come in different sizes, from stellar-mass to supermassive black holes. Their gravity is caused by immense mass, with event horizons from which nothing can escape. They distort time and space, exhibit Hawking radiation, and provide insights into gravity and extreme conditions. Despite their mysterious nature, black holes are not as dangerous as portrayed and continue to be a focus of research in astrophysics.


- So when we are using `StrOutputParser`, we don't have to explicitly print `.content` as we had to earlier.
- Also, we can use chain to send output of first model to parser and from parser to second model, less boiler plate code.
- Works both for OpenAI and HuggingFace models.

### Flaw:
- This just extract the `content` part from generated output. No restructuring according to usecase.
- Restructuring can be done via `JsonOutputParser` - will see next.

## JsonOutputParser

### HuggingFace Model - Example1

In [ ]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser


load_dotenv()


# can try with "TinyLlama/TinyLlama-1.1B-chat-v1.0" as well
llm = HuggingFaceEndpoint(
    repo_id = "google/gemma-2-2b-it",
    tasks = "text-generation"
)

model = ChatHuggingFace(llm=llm)

parser = JsonOutputParser()

template = PromptTemplate(
    template = "Give me the name, age and city of a fictional person \n {format_instruction}",
    input_variables = [],
    partial_variables = {'format_instruction': parser.get_format_instructions()}
)

- `partial_variables`: It is called `partial_variables` because it doesn't fail at runtime, it fails before runtime for the function `get_format_instructions`.
- `input_variables` as there is no runtime variable to pass.

In [ ]:
prompt = template.format() # static prompt
print(prompt)

result = model.invoke(prompt)
print(result)

final_result = parser.parse(result.content)

print(final_result)
print(type(final_result))

- This is the descriptive way where is boiler plate code present to work on steps.
- Here the `format` function needs no input as there is no input to pass at runtime. It's called static prompt, not dynamic prompt, though class `PromptTemplate` is used. It's used to give partial_input in the prompt, not to make it dynamic.

In [ ]:
chain = template | model | parser
result = chain.invoke({}) 
print(result) 

- This is usning `chain` where automatically the pipeline runs without boiler plate code.
- Also, as no input to send at runtime, `invoke` function has no input to pass.
- Here the prompt is static prompt, as mentioned in descriptive code and no runtime input. 

### HuggingFace Model - Example2

In [ ]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser


load_dotenv()


# can try with "TinyLlama/TinyLlama-1.1B-chat-v1.0" as well
llm = HuggingFaceEndpoint(
    repo_id = "google/gemma-2-2b-it",
    tasks = "text-generation"
)

model = ChatHuggingFace(llm=llm)

parser = JsonOutputParser()

template = PromptTemplate(
    template = "Give me 5 facts about {topic} \n {format_instruction}",
    input_variables = ['topic'],
    partial_variables = {'format_instruction': parser.get_format_instructions()}
)

- Here is along with `partial_input`, another runtime input `topic` unlike earlier eaxmple where there was no runtime input.
- This is dynamic prompt here, not static like earlier example.

In [ ]:
prompt = template.format(topic = 'black hole') # dynamic prompt
print(prompt)

result = model.invoke(prompt)
print(result)

final_result = parser.parse(result.content)

print(final_result)
print(type(final_result))

- This is the descriptive way where is boiler plate code present to work on steps.

In [ ]:
chain = template | model | parser
result = chain.invoke({'topic': 'black hole'})
print(result)

- This is usning `chain` where automatically the pipeline runs without boiler plate code.
- `chain` can be used post template generation as well.
- Also, as input `topic` has to be sent at runtime, we sent `black hole` in `invoke` function.
- Here the prompt is dynamic prompt, as mentioned in descriptive code and there is runtime input. 

In [ ]:
chain = template | model | parser
result = chain.invoke({}) 
print(result)

- Above code will throw error as no input `topic` is provided.
- Similarly this parser works for OpenAI model, just normal change is at the line of LLM creation.

### Flaw:
- This output parser doesn't enforce json schema (if i need output in a particular custom format, can't be done via jsonoutputparser)
- schema enforcing can be done with `StructuredOutputParser` - will see next.

## StructuredOutputParser

### HuggingFace Model

In [ ]:
# another example with input
# working with HuggingFace (these can't return work with with_structured_output)
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain.output_parsers import StructuredOutputParser, ResponseSchema


load_dotenv()


# can try with "TinyLlama/TinyLlama-1.1B-chat-v1.0" as well
llm = HuggingFaceEndpoint(
    repo_id = "google/gemma-2-2b-it",
    tasks = "text-generation"
)

model = ChatHuggingFace(llm=llm)

schema = [
    ResponseSchema(name='fact_1', description = 'Fact 1 about the topic'),
    ResponseSchema(name='fact_2', description = 'Fact 2 about the topic'),
    ResponseSchema(name='fact_3', description = 'Fact 3 about the topic'),
    ResponseSchema(name='fact_4', description = 'Fact 4 about the topic'),
    ResponseSchema(name='fact_5', description = 'Fact 5 about the topic'),
]

parser = StructuredOutputParser.from_response_schemas(schema)

template = PromptTemplate(
    template = 'Give 5 facts about {topic} \n {format_instructions}',
    input_variables = ['topic'],
    partial_variables = {'format_instruction': parser.get_format_instructions()}
)

In [ ]:
prompt = template.invoke({'topic': 'black hole'})
result = model.invoke(prompt)
final_result = parser.parse(result.content)

print(final_result)

- This is the descriptive way.
- Input related descriptions same as discussed earlier.

In [ ]:
chain = template | model | parser
result = chain.invoke({'topic': 'black hole'})
print(result)

- This is using `chain`.
- Input related descriptions same as discussed earlier.

- Similarly this works for OpenAI model, just normal change is at the line of LLM creation.

### Flaw:
- Data validation (enforcing datatype and format of output fields) can't be done in this one, e.g. I want 'year': 35 but model sends 'year': '35 years' or '35' (i.e. instead of integerm LLM sends string), but at least schema can be enforced.
- Data validation can be done via `PydanticOutputParser` - will see next.

## PydanticOutputParser

### HuggingFace Model

In [ ]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

load_dotenv()

# can try with "TinyLlama/TinyLlama-1.1B-chat-v1.0" as well
llm = HuggingFaceEndpoint(
    repo_id = "google/gemma-2-2b-it",
    tasks = "text-generation"
)

model = ChatHuggingFace(llm=llm)

class Person(BaseModel):
    name: str = Field(description="name of the person")
    age: int = Field(gt=18, description='Age of the person')
    city: int = Field(description='Name of the city person beongs to')

parser = PydanticOutputParser(pydantic_object=Person)
template = PromptTemplate(
    template = 'Generate the name, age anf city of a fictional {place} person \n {format_instruction}',
    input_variables = ['place'],
    partial_variables = {'format_instruction': parser.get_format_instructions()}
)

prompt = template.invoke({'place': 'indian'})
print("The generated prompt is:\n", prompt)
print()

result = model.invoke(prompt)
final_result = parser.parse(result.content)

print(final_result)

- This is the descriptive way.
- Input related descriptions same as discussed earlier.

In [ ]:
chain = template | model | parser
result = chain.invoke({'place': 'sri lankan'})
print(result)

- This is using `chain`.
- Input related descriptions same as discussed earlier.

- Similarly this works for OpenAI model, just normal change is at the line of LLM creation.

## Other Parsers
- There are few more parsers in LangChain documentation.
- These four are mostly used indifferent usecases.
- For any usecase, we have to check different parser from the documentation.